In [146]:
import numpy as np
import CircuitBuilder
from tqdm import tqdm

In [147]:
defaultzschedule = np.asarray([
    [1, 2, 3, 4, 0, 0, 0, 0],
    [0, 0, 0, 0, 1, 2, 3, 4],
    [2, 3, 0, 0, 4, 1, 0, 0],
    [3, 0, 4, 0, 2, 0, 1, 0]
])
defaultxschedule = np.asarray([
    [1, 2, 3, 4, 5, 6, 7, 8]
])

def windowFiller(schedule, nwindow):
    thiswindow = []
    for row in schedule:
        if nwindow in row:
            thiswindow.append(np.where(row == nwindow)[0][0])
        else:
            thiswindow.append(None)
    return thiswindow

def c832scheduleableinjectable2(er = 0, nrounds = 1, zschedule = defaultzschedule, xschedule = defaultxschedule, errorschedule = None):
    zschedule, xschedule = np.asarray(zschedule), np.asarray(xschedule)
    #cb = CircuitBuilder.CircuitBuilder(er, er, er, er, er, 0)
    cb = CircuitBuilder.CircuitBuilder(er, er, er, er, er, er)
    #cb = CircuitBuilder.CircuitBuilder(er, er, er, er, 0, er)
    #cb = CircuitBuilder.CircuitBuilder(er, er, er, 0, 0, 0)
    cb.al('R 0 1 2 3 4 5 6 7')
    cb.al('H 0 1 2 3 4 5 6 7')
    nancillas = len(zschedule) + len(xschedule)
    ancillastring = 'R ' + ' '.join([f'{i}' for i in range(8, 8 + nancillas)])
    cb.al(ancillastring)
    cb.set_ancilla_qubits(qubitslistz = [8 + i for i in range(len(zschedule))], qubitslistx = [8 + len(zschedule) + i for i in range(len(xschedule))])
    #print(cb.ancilla_qubitsz, cb.ancilla_qubitsx)
    # Warmup round
    for windowz in range(1, np.max(zschedule) + 1):
        #cb.time_tick()
        #print(windowFiller(zschedule, windowz))
        cb.couple_z_check(windowFiller(zschedule, windowz), False)
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    for windowx in range(1, np.max(xschedule) + 1):
        #cb.time_tick()
        cb.couple_x_check(windowFiller(xschedule, windowx), False)
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    # Noise rounds

    timewindow = 0
    for _ in range(nrounds):
        for windowz in range(1, np.max(zschedule) + 1):
            #cb.time_tick()
            #cb.al('TICK')
            if errorschedule is not None:
                for errornum in range(len(errorschedule[timewindow])):
                    if errorschedule[timewindow][errornum] != 'I':
                        cb.inject_error([errornum], errorschedule[timewindow][errornum])
            timewindow += 1
            cb.couple_z_check(windowFiller(zschedule, windowz), True)
        cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
        cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
        cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
        for windowx in range(1, np.max(xschedule) + 1):
            #cb.time_tick()
            #cb.al('TICK')
            if errorschedule is not None:
                for errornum in range(len(errorschedule[timewindow])):
                    if errorschedule[timewindow][errornum] != 'I':
                        cb.inject_error([errornum], errorschedule[timewindow][errornum])
            timewindow += 1
            cb.couple_x_check(windowFiller(xschedule, windowx), True)
        cb.add_detectors([[-1 - i, -1 - i - nancillas] for i in range(len(cb.ancilla_qubitsz))])
        cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
        cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
        cb.add_detectors([[-1 - i, -1 - i - nancillas] for i in range(len(cb.ancilla_qubitsx))])
        cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    # Cooldown round
    for windowz in range(1, np.max(zschedule) + 1):
        cb.couple_z_check(windowFiller(zschedule, windowz), False)
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    for windowx in range(1, np.max(xschedule) + 1):
        cb.couple_x_check(windowFiller(xschedule, windowx), False)
    cb.add_detectors([[-1 - i, -1 - i - nancillas] for i in range(len(cb.ancilla_qubitsz))])
    cb.al('H ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.al('MR(0) ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsx]))
    cb.add_detectors([[-1 - i, -1 - i - nancillas] for i in range(len(cb.ancilla_qubitsx))])
    cb.al('R ' + ' '.join([f'{i}' for i in cb.ancilla_qubitsz]))
    # Final measurement of all physical qubits
    cb.al('H 0 1 2 3 4 5 6 7')
    cb.al(f'MR({0}) 0 1 2 3 4 5 6 7')
    return cb

def detector_builder(mseq, syndromelength = 5, nphysicalqubits = 8, nrounds = 1, mode = 'binary'):
    if len(mseq) != syndromelength * (nrounds + 1) + nphysicalqubits:
        raise ValueError(f"Measurement sequence length must be equal to syndromelength * nrounds + nphysicalqubits")
    syndromes, finalmeasurement = mseq[:-nphysicalqubits], mseq[-nphysicalqubits:]
    if mode == 'string':
        det = []
        for i in range(nrounds):
            det.append(''.join([str(0 if x == y else 1) for x, y in zip(syndromes[i*syndromelength:(i+1)*syndromelength], syndromes[(i+1)*syndromelength:(i+2)*syndromelength])]))
    elif mode == 'binary':
        det = np.zeros(nrounds * syndromelength, dtype = np.bool)
        for i in range(nrounds):
            det[i*syndromelength:(i+1)*syndromelength] = np.array([0 if x == y else 1 for x, y in zip(syndromes[i*syndromelength:(i+1)*syndromelength], syndromes[(i+1)*syndromelength:(i+2)*syndromelength])])
    else:
        raise ValueError(f"Invalid mode: {mode}")

    return det, finalmeasurement

def check_detector_fired(det):
    for detector in det:
        if '1' in detector:
            return True
    return False

def measure_readout(readout, logical_operator):
    parity = 0
    for i in range(len(readout)):
        if readout[i] == '1' and logical_operator[i]:
            parity += 1
    return parity % 2


In [148]:
zschedule1 = [[3, 4, 1, 2, 0, 0, 0, 0],
 [0, 0, 0, 0, 4, 3, 2, 1],
 [1, 3, 0, 0, 2, 4, 0, 0],
 [4, 0, 2, 0, 1, 0, 3, 0]]
xschedule1 = [[4, 0, 1, 0, 0, 3, 0, 2],
 [3, 0, 4, 0, 1, 0, 2, 0],
 [0, 1, 0, 2, 4, 0, 3, 0]]
zschedule2 = [[1, 4, 2, 3, 0, 0, 0, 0],
 [0, 0, 0, 0, 2, 3, 1, 4],
 [3, 2, 0, 0, 4, 1, 0, 0],
 [2, 0, 3, 0, 1, 0, 4, 0]]
xschedule2 = [[1, 0, 4, 0, 0, 3, 0, 2],
 [2, 0, 3, 0, 4, 0, 1, 0],
 [0, 1, 0, 4, 3, 0, 2, 0]]
logop = np.array([1, 1, 0, 0, 1, 1, 0, 0])

In [155]:
perror, nshots, nrounds = 0.02, 1, 1
errorschedulelength = nrounds * 8
errors = []
readoutdict = {}
s, sandc = 0, 0
emptyerrorschedule = [
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I']
    for _ in range(errorschedulelength)
]
#zschedule, xschedule = zschedule1, xschedule1
zschedule, xschedule = zschedule2, xschedule1
simulated = 0
for _ in tqdm(range(10000)):
    numerrors = 0
    errorschedule = emptyerrorschedule.copy()
    thiserrors = []
    for rowcol in range((errorschedulelength) * 15):
        row, col = rowcol // 15, rowcol % 15
        randomerror = np.random.choice(['I','X', 'Z', 'Y'], p = [1-perror, perror/3, perror/3, perror/3])
        errorschedule[row][col] = randomerror
        if randomerror != 'I':
            numerrors += 1
            thiserrors.append((row, col, randomerror))
    if numerrors != 1:
        continue
    simulated += 1
    cb = c832scheduleableinjectable2(er = 0, nrounds = nrounds, zschedule = zschedule, xschedule = xschedule, errorschedule = errorschedule)
    sample = cb.sample(nshots)
    count = 0
    for row in sample:
        det, finalmeasurement = detector_builder(row[0], 7, 8, nrounds + 1, mode = 'string')
        correctreadout = measure_readout(finalmeasurement, logop) == 0
        survives = not check_detector_fired(det)
        if survives:
            s += row[1]
            if finalmeasurement not in readoutdict:
                readoutdict[finalmeasurement] = row[1]
            else:
                readoutdict[finalmeasurement] += row[1]
        if survives and correctreadout:
            sandc += row[1]
        if survives and not correctreadout:
            errors.append(thiserrors)
print(simulated, len(errors))
for row in errors:
    print(row)

100%|██████████| 10000/10000 [00:11<00:00, 850.74it/s]

2190 10
[(2, 8, np.str_('Z'))]
[(2, 9, np.str_('Z'))]
[(2, 8, np.str_('Z'))]
[(2, 9, np.str_('Z'))]
[(2, 9, np.str_('Z'))]
[(2, 8, np.str_('Z'))]
[(2, 9, np.str_('Z'))]
[(2, 9, np.str_('Z'))]
[(2, 8, np.str_('Z'))]
[(2, 8, np.str_('Z'))]


In [152]:
# For custom error injections
customerrorschedule = [
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'Z', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
    ['I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I', 'I'],
]
if False:
    errorcount = 0
    perror = 0.01
    for i in range(15 * len(customerrorschedule)):
        row, col = i // 15, i % 15
        customerrorschedule[row][col] = np.random.choice(['I', 'X', 'Z', 'Y'], p = [1-perror, perror/3, perror/3, perror/3])
        errorcount += 1 if customerrorschedule[row][col] != 'I' else 0
    print(errorcount)
    for row in customerrorschedule:
        print(row)
    
#zschedule, xschedule = zschedule1, xschedule1
zschedule, xschedule = zschedule2, xschedule1
errorprob = 0
cb = c832scheduleableinjectable2(er = 0, nrounds = 1, zschedule = zschedule, xschedule = xschedule, errorschedule = customerrorschedule)
sample = cb.sample(1)
det, finalmeasurement = detector_builder(sample[0][0], 7, 8, 1+1, mode = 'string')
print(sample)
print(det)
print(finalmeasurement)
print(measure_readout(finalmeasurement, logop))

[('01100000110000011000010010011', 1)]
['0000000', '0000000']
10010011
1


In [157]:
cb.get_circuit().diagram()

q0: -R-H-@-------------@-----@--------------------------------------X---X--------------------@---------------@-----@-----------------------------------------X---X-------------------------------------------------------------------------@-------------@-----@-----------------------------------------X---X-----------------------------------------------H----------------------------MR(0):rec[21]-
          |             |     |                                      |   |                    |               |     |                                         |   |                                                                         |             |     |                                         |   |
 q1: -R-H-|-----------@-|-----|---@------------------------X---------|---|--------------------|-------------@-|-----|---@---------------------------X---------|---|-------------------------------------------------------------------------|-----------@-|-----|---@---------------------------X---------|---|-----------------------------------------------H----------------------------MR(0):rec[22]-
          |           | |     |   |                        |         |   |                    |             | |     |   |                           |         |   |                                                                         |           | |     |   |                           |         |   |
 q2: -R-H-|-------@---|-|-----|-@-|--------------------X---|---------|---|-X------------------|---------@---|-|-----|-@-|-----------------------X---|---------|---|-X-----------------------------------------------------------------------|-------@---|-|-----|-@-|-----------------------X---|---------|---|-X---------------------------------------------H----------------------------MR(0):rec[23]-
          |       |   | |     | | |                    |   |         |   | |                  |         |   | |     | | |                       |   |         |   | |                                                                       |       |   | |     | | |                       |   |         |   | |
 q3: -R-H-|-------|---|-|-@---|-|-|--------------------|---|-----X---|---|-|------------------|---------|---|-|-@---|-|-|-----------------------|---|-----X---|---|-|-----------------------------------------------------------------------|-------|---|-|-@---|-|-|-----------------------|---|-----X---|---|-|---------------------------------------------H----------------------------MR(0):rec[24]-
          |       |   | | |   | | |                    |   |     |   |   | |                  |         |   | | |   | | |                       |   |     |   |   | |                                                                       |       |   | | |   | | |                       |   |     |   |   | |
 q4: -R-H-|-----@-|-@-|-|-|---|-|-|---@----------------|-X-|-----|---|---|-|-X----------------|-----@---|-@-|-|-|---|-|-|---@-------------------|-X-|-----|---|---|-|-X---------------------------------------------------------------------|-----@-|-@-|-|-|---|-|-|---@-------------------|-X-|-----|---|---|-|-X-------------------------------------------H----------------------------MR(0):rec[25]-
          |     | | | | | |   | | |   |                | | |     |   |   | | |                |     |   | | | | |   | | |   |                   | | |     |   |   | | |                                                                     |     | | | | | |   | | |   |                   | | |     |   |   | | |
 q5: -R-H-|---@-|-|-|-|-|-|-@-|-|-|---|----------------|-|-|-----|-X-|---|-|-|----------------|---@-|---|-|-|-|-|-@-|-|-|---|-------------------|-|-|-----|-X-|---|-|-|---------------------------------------------------------------------|---@-|-|-|-|-|-|-@-|-|-|---|-------------------|-|-|-----|-X-|---|-|-|-------------------------------------------H----------------------------MR(0):rec[26]-
          |   | | | | | | | | | | |   |                | | |     | | |   | | |                |   | |   | | |